# LSTM Recommender System

This notebook implements a LSTM-based (unidirectional) sequential recommender using user interaction sequences.

## Architecture:
- Input: User interaction sequences (post embeddings)
- LSTM layer (Unidirectional)
- Global Average Pooling
- Output: Next item prediction

## Data Flow:
1. Load posts and user interaction data
2. Create temporal user interaction sequences
3. Generate post embeddings (TF-IDF features with caption + description)
4. Train LSTM model
5. Evaluate using Hit Rate and MRR

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.sequence import pad_sequences

# NLP & Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# Metrics
from sklearn.metrics.pairwise import cosine_similarity

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Load Data

In [ ]:
# Load data
interactions_df = pd.read_csv('input/interaction_sequences.csv')
posts_df = pd.read_csv('data/clean/clean_posts.csv')

print(f"Interactions: {interactions_df.shape}")
print(f"Posts: {posts_df.shape}")

print("\n=== Sample Data ===")
display(interactions_df.head(3))
display(posts_df[['post_id', 'caption']].head(3))

## 3. Create User Interaction Sequences

In [ ]:
def prepare_user_sequences(interactions_df):
    """Parse posts_sequence from CSV."""
    import ast
    
    interactions_df['posts_sequence_parsed'] = interactions_df['posts_sequence'].apply(
        lambda x: ast.literal_eval(x) if pd.notna(x) else []
    )
    
    df_filtered = interactions_df[interactions_df['sequence_length'] > 2].copy()
    
    return df_filtered[['commentUser', 'posts_sequence_parsed']].rename(
        columns={'posts_sequence_parsed': 'posts_sequence'}
    )

df_sequence = prepare_user_sequences(interactions_df)
print(f"Users with >2 interactions: {df_sequence.shape[0]}")

print("\nSample sequences:")
for i in range(min(3, len(df_sequence))):
    user = df_sequence.iloc[i]['commentUser']
    seq = df_sequence.iloc[i]['posts_sequence']
    print(f"  {user}: {len(seq)} posts -> {seq[:5]}{'...' if len(seq) > 5 else ''}")

display(df_sequence.head())

In [ ]:
# Filter users with >2 interactions for meaningful sequences
df_sequence = df_sequence[df_sequence['posts_sequence'].apply(lambda x: len(x) > 2)]

print(f"Users with >2 interactions: {df_sequence.shape[0]}")
print(f"\nSequence length statistics:")
seq_lengths = df_sequence['posts_sequence'].apply(len)
print(f"  Min: {seq_lengths.min()}")
print(f"  Max: {seq_lengths.max()}")
print(f"  Mean: {seq_lengths.mean():.2f}")
print(f"  Median: {seq_lengths.median():.0f}")

display(df_sequence.head())

Users with >2 interactions: 0

Sequence length statistics:


KeyError: 'posts_sequence'

## 4. Create Post Embeddings

We'll create post embeddings using:
- TF-IDF on captions (text features)
- Image embeddings (if available)
- Combined into a single feature vector

In [ ]:
# Create post features from captions
posts_df['caption_clean'] = posts_df['caption'].fillna('')

# TF-IDF
tfidf = TfidfVectorizer(max_features=256, ngram_range=(1, 2), min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(posts_df['caption_clean'])
tfidf_features = normalize(tfidf_matrix.toarray(), axis=1)

print(f"TF-IDF features shape: {tfidf_features.shape}")

# Sample features
for i in range(min(3, len(posts_df))):
    caption = posts_df.iloc[i]['caption_clean']
    print(f"  Post {posts_df.iloc[i]['post_id']}: {caption[:100]}...")

print(f"\n✓ Using Caption features (rich user-generated text)")

In [ ]:
# Note: Image descriptions are already included in the text_features above
# We're using TF-IDF features that combine:
# - Image descriptions (from fashion_description.csv)
# - Fashion attributes (color, category, style, occasion)
# - Keywords from posts

print("✓ Using integrated features (text + image descriptions + fashion attributes)")
print(f"Feature dimension: {tfidf_features.shape[1]}")

In [ ]:
# Create feature mapping: post_id -> embedding
post_features = {}
for idx, post_id in enumerate(posts_df['post_id']):
    post_features[post_id] = tfidf_features[idx]

FEATURE_DIM = tfidf_features.shape[1]
print(f"Feature dimension: {FEATURE_DIM}")
print(f"Total posts with features: {len(post_features)}")

## 5. Prepare Training Data

In [ ]:
# Validate sequences: remove posts not in feature mapping
def validate_sequence(seq):
    return [p for p in seq if p in post_features]

df_sequence['posts_sequence'] = df_sequence['posts_sequence'].apply(validate_sequence)
df_sequence = df_sequence[df_sequence['posts_sequence'].apply(lambda x: len(x) > 2)]

print(f"Validated sequences: {df_sequence.shape[0]}")

In [ ]:
# Hyperparameters
WINDOW_SIZE = 10  # Look at last 10 interactions
HIDDEN_DIM = 128
L2_REG = 1e-5

print(f"Window size: {WINDOW_SIZE}")
print(f"Hidden dimension: {HIDDEN_DIM}")
print(f"L2 regularization: {L2_REG}")

In [ ]:
def create_training_samples(sequences, post_features, window_size=10):
    """
    Create (X, y) pairs for training.
    X: sequence of embeddings (window_size)
    y: next post_id
    """
    X_sequences = []
    y_next_posts = []
    
    for seq in sequences:
        if len(seq) < 2:
            continue
        
        # Sliding window
        for i in range(1, len(seq)):
            history = seq[max(0, i-window_size):i]
            next_post = seq[i]
            
            # Convert to embeddings
            history_embeddings = [post_features[p] for p in history]
            
            # Pad if needed
            while len(history_embeddings) < window_size:
                history_embeddings.insert(0, np.zeros(FEATURE_DIM))
            
            X_sequences.append(history_embeddings[:window_size])
            y_next_posts.append(next_post)
    
    return np.array(X_sequences), np.array(y_next_posts)

# Create training data
sequences = df_sequence['posts_sequence'].tolist()
X_train, y_train = create_training_samples(sequences, post_features, WINDOW_SIZE)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## 6. Build LSTM Model

In [ ]:
def build_lstm_model(window_size, feature_dim, hidden_dim, l2_reg):
    """
    Build LSTM model (unidirectional).
    
    Architecture:
    - Input: (batch, window_size, feature_dim)
    - LSTM: (batch, window_size, hidden_dim)
    - Global Average Pooling: (batch, hidden_dim)
    - Dense: (batch, feature_dim) for next item prediction
    """
    # Input
    sequence_input = layers.Input(shape=(window_size, feature_dim), name='sequence_input')
    
    # LSTM layer (unidirectional)
    lstm_out = layers.LSTM(
        hidden_dim,
        return_sequences=True,
        kernel_regularizer=regularizers.l2(l2_reg),
        name='lstm'
    )(sequence_input)
    
    # Global pooling (aggregate sequence)
    pooled = layers.GlobalAveragePooling1D(name='global_pool')(lstm_out)
    
    # Dropout
    dropout = layers.Dropout(0.3, name='dropout')(pooled)
    
    # Output layer: predict next item embedding
    output = layers.Dense(
        feature_dim,
        activation=None,
        kernel_regularizer=regularizers.l2(l2_reg),
        name='output'
    )(dropout)
    
    # L2 normalize output for cosine similarity
    output_normalized = layers.Lambda(lambda x: tf.nn.l2_normalize(x, axis=1), name='normalize')(output)
    
    model = models.Model(inputs=sequence_input, outputs=output_normalized, name='LSTM')
    
    return model

# Build model
model = build_lstm_model(
    window_size=WINDOW_SIZE,
    feature_dim=FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    l2_reg=L2_REG
)

model.summary()

## 7. Prepare Target Embeddings

In [ ]:
# Create target embeddings for y_train
y_train_embeddings = np.array([post_features[post_id] for post_id in y_train])

# Normalize
y_train_embeddings = normalize(y_train_embeddings, axis=1)

print(f"y_train_embeddings shape: {y_train_embeddings.shape}")

## 8. Compile and Train Model

In [ ]:
# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='cosine_similarity',  # Negative cosine similarity
    metrics=['mae']
)

print("Model compiled successfully")

In [ ]:
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Train model
history = model.fit(
    X_train,
    y_train_embeddings,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 9. Evaluate Model

In [ ]:
def evaluate_recommendations(model, test_sequences, post_features, k=5):
    """
    Evaluate using Hit Rate @ K and MRR.
    """
    hits = 0
    mrr_sum = 0
    total = 0
    
    all_post_ids = list(post_features.keys())
    all_embeddings = np.array([post_features[p] for p in all_post_ids])
    all_embeddings = normalize(all_embeddings, axis=1)
    
    for seq in test_sequences:
        if len(seq) < 2:
            continue
        
        # Use last items as test
        history = seq[:-1]
        ground_truth = seq[-1]
        
        # Prepare input
        history_window = history[-WINDOW_SIZE:]
        history_embeddings = [post_features[p] for p in history_window]
        
        # Pad if needed
        while len(history_embeddings) < WINDOW_SIZE:
            history_embeddings.insert(0, np.zeros(FEATURE_DIM))
        
        X_test = np.array([history_embeddings[:WINDOW_SIZE]])
        
        # Predict
        pred_embedding = model.predict(X_test, verbose=0)[0]
        
        # Compute similarity with all posts
        similarities = cosine_similarity([pred_embedding], all_embeddings)[0]
        
        # Get top-k recommendations
        top_k_indices = np.argsort(similarities)[::-1][:k]
        top_k_posts = [all_post_ids[i] for i in top_k_indices]
        
        # Check hit
        if ground_truth in top_k_posts:
            hits += 1
            # Calculate MRR
            rank = top_k_posts.index(ground_truth) + 1
            mrr_sum += 1.0 / rank
        
        total += 1
    
    hit_rate = hits / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    return hit_rate, mrr, total

# Evaluate on all sequences
hit_rate_5, mrr, total = evaluate_recommendations(model, sequences, post_features, k=5)

print(f"\n=== Evaluation Results ===")
print(f"Hit Rate @ 5: {hit_rate_5:.4f}")
print(f"MRR: {mrr:.4f}")
print(f"Total test cases: {total}")

## 10. Save Model

In [ ]:
# Save model
model.save('models/lstm_model.h5')
print("Model saved to models/lstm_model.h5")

# Save post features
import pickle
with open('models/post_features.pkl', 'wb') as f:
    pickle.dump(post_features, f)
print("Post features saved to models/post_features.pkl")

## 11. Inference Example

In [ ]:
def recommend_next_posts(user_history, model, post_features, k=5):
    """
    Recommend next k posts given user history.
    
    Args:
        user_history: List of post_ids
        model: Trained BiLSTM model
        post_features: Dict of post_id -> embedding
        k: Number of recommendations
    """
    # Prepare input
    history_window = user_history[-WINDOW_SIZE:]
    history_embeddings = [post_features[p] for p in history_window if p in post_features]
    
    if len(history_embeddings) == 0:
        return []
    
    # Pad if needed
    while len(history_embeddings) < WINDOW_SIZE:
        history_embeddings.insert(0, np.zeros(FEATURE_DIM))
    
    X_input = np.array([history_embeddings[:WINDOW_SIZE]])
    
    # Predict
    pred_embedding = model.predict(X_input, verbose=0)[0]
    
    # Get all posts
    all_post_ids = list(post_features.keys())
    all_embeddings = np.array([post_features[p] for p in all_post_ids])
    all_embeddings = normalize(all_embeddings, axis=1)
    
    # Compute similarity
    similarities = cosine_similarity([pred_embedding], all_embeddings)[0]
    
    # Get top-k
    top_k_indices = np.argsort(similarities)[::-1][:k]
    recommendations = [(all_post_ids[i], similarities[i]) for i in top_k_indices]
    
    return recommendations

# Example: recommend for first user
example_user = df_sequence.iloc[0]
user_history = example_user['posts_sequence'][:-1]  # Exclude last item
ground_truth = example_user['posts_sequence'][-1]

print(f"User: {example_user['commentUser']}")
print(f"History: {user_history[-5:]}")
print(f"Ground truth: {ground_truth}")
print("\nRecommendations:")

recommendations = recommend_next_posts(user_history, model, post_features, k=5)
for i, (post_id, score) in enumerate(recommendations, 1):
    is_correct = "✓" if post_id == ground_truth else ""
    print(f"{i}. {post_id} (score: {score:.4f}) {is_correct}")